
# Optimizing Neural Networks

Training a neural network is an optimization problem. We want to reach the minimum error as fast as possible without getting stuck. Basic Gradient Descent is often too slow or unstable for deep networks. To fix this, we use advanced **Optimization** and **Regularization** techniques.

## Weight Initialization: Starting Right

If we initialize all weights to Zero, every neuron learns the same thing (symmetry problem). The network becomes a linear model. If we initialize them too large, gradients explode. Too small, gradients vanish.

**Strategies**:

-   **He Initialization**: Best for ReLU networks. Keeps the variance of activations constant across layers.
-   **Xavier (Glorot) Initialization**: Best for Sigmoid/Tanh networks.

## Optimizers: Smarter Descent

Standard SGD is like walking down a mountain blindfolded. Advanced optimizers add "momentum" or adaptive steps.

| Optimizer          | Mechanism                                               | Analogy                                                 |
|------------------ |------------------------------------------------------- |------------------------------------------------------- |
| **SGD + Momentum** | Accumulates past gradients to smooth out the path.      | A heavy ball rolling downhill (builds speed).           |
| **RMSProp**        | Adapts the learning rate for each parameter separately. | Slows down on steep slopes, speeds up on flat plains.   |
| **Adam**           | Combines Momentum + RMSProp. The default choice.        | A smart ball that adapts speed and direction perfectly. |

## Regularization: Fighting Overfitting

Deep networks are prone to memorizing data. We need techniques to force them to generalize.

### Dropout

Randomly "kill" (set to zero) a percentage of neurons during training. This prevents neurons from co-adapting and relying too much on specific features. It forces the network to be redundant and robust.

### Batch Normalization

Normalizes the inputs of each layer to have mean 0 and variance 1 **during training**.

-   Stabilizes training (allows higher learning rates).
-   Reduces sensitivity to initialization.

### Early Stopping

Monitor the Validation Loss. If it stops improving (or starts getting worse), stop training immediately. This prevents the model from overfitting in the late stages.

## Practical Demonstration: Adam vs SGD

We will train a network on the "Digits" dataset using standard SGD versus Adam to see the speed difference.

### Setup Data

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Load data
digits = load_digits()
X, y = digits.data, digits.target

# Split first, then scale using training data only (prevents data leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Training Comparison

We train two identical MLPs, changing only the `solver`.

In [ ]:
from sklearn.neural_network import MLPClassifier

# SGD Model
mlp_sgd = MLPClassifier(hidden_layer_sizes=(64, 32), 
                        solver='sgd', 
                        learning_rate_init=0.01, 
                        max_iter=200, 
                        random_state=42)

# Adam Model
mlp_adam = MLPClassifier(hidden_layer_sizes=(64, 32), 
                         solver='adam', 
                         learning_rate_init=0.01, 
                         max_iter=200, 
                         random_state=42)

mlp_sgd.fit(X_train, y_train)
mlp_adam.fit(X_train, y_train)

print(f"SGD Accuracy:  {mlp_sgd.score(X_test, y_test):.3f}")
print(f"Adam Accuracy: {mlp_adam.score(X_test, y_test):.3f}")

### Visualizing Convergence Speed

We plot the Loss Curves. Adam usually drops much faster.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(mlp_sgd.loss_curve_, label='SGD', color='red', linestyle='--')
plt.plot(mlp_adam.loss_curve_, label='Adam', color='blue')
plt.title("Convergence Speed: Adam vs SGD")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.legend()
plt.show()

## Practical Demonstration: Early Stopping

We will force a network to overfit and see if Early Stopping saves us.

In [ ]:
# Baseline model WITHOUT Early Stopping
mlp_no_early = MLPClassifier(hidden_layer_sizes=(100,),
                             max_iter=500,
                             early_stopping=False,
                             random_state=42)

# Model WITH Early Stopping
# early_stopping=True sets aside validation data for monitoring
mlp_early = MLPClassifier(hidden_layer_sizes=(100,), 
                          max_iter=500, 
                          early_stopping=True, 
                          validation_fraction=0.1,
                          n_iter_no_change=10, 
                          random_state=42)

mlp_no_early.fit(X_train, y_train)
mlp_early.fit(X_train, y_train)

print(f"No Early Stopping - Iterations: {mlp_no_early.n_iter_}, Test Acc: {mlp_no_early.score(X_test, y_test):.3f}")
print(f"With Early Stopping - Iterations: {mlp_early.n_iter_}, Test Acc: {mlp_early.score(X_test, y_test):.3f}")
print(f"Best Validation Score: {mlp_early.best_validation_score_:.3f}")

# Plot training losses together; show validation score separately
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].plot(mlp_no_early.loss_curve_, label='No Early Stopping', color='gray')
ax[0].plot(mlp_early.loss_curve_, label='With Early Stopping', color='blue')
ax[0].set_title("Training Loss Comparison")
ax[0].set_xlabel("Iterations")
ax[0].set_ylabel("Loss")
ax[0].legend()

ax[1].plot(mlp_early.validation_scores_, label='Validation Accuracy', color='green')
ax[1].set_title("Early Stopping Validation Monitor")
ax[1].set_xlabel("Iterations")
ax[1].set_ylabel("Validation Accuracy")
ax[1].legend()

plt.tight_layout()
plt.show()

## Exercises

**Note**: The exercises below all use the Wine dataset.

### The Effect of Feature Scaling

This exercise demonstrates why **feature scaling** matters for optimization using a dataset with naturally heterogeneous feature scales.

1.  Load the Wine dataset.
2.  Train an MLP on scaled features.
3.  Train the same MLP on unscaled features.
4.  Compare the loss curves and test accuracy.

### Regularization Strength (Alpha)

Using the same Wine dataset, the `alpha` parameter controls L2 Regularization (Weight Decay).

-   Train three models with different `alpha` values.
-   Compare train/test accuracy and total weight norm.
-   Interpret which model appears underfit vs. better balanced.

**Prompt**: Which alpha gives the best trade-off? Look for smaller weights with minimal drop in test accuracy.

## Summary

1.  **Adam**: generally the best default optimizer. Fast and robust.
2.  **Early Stopping**: A practical way to limit overfitting, with light tuning (e.g., patience/validation settings).
3.  **Normalization**: Neural networks usually train faster and more stably with scaled inputs.